# INDILEX Batch Annotation Notebook — Indian Legal Judgments (Groq API)

Production-grade, case-type-aware, definition-driven batch annotator for **Criminal, Civil, Constitutional, and Administrative** judgments.

## How to use

1. Put your Groq API key in a `.env` file next to this notebook (or in your environment):
   ```
   GROQ_API_KEY=gsk_...
   ```
2. In **CELL 4 (USER CONFIGURATION)** set only:
   - `INPUT_FILE`  — the batch CSV to annotate
   - `OUTPUT_DIR`  — where annotated output, checkpoint, error report, duplicate report and log are written
3. Run all cells top to bottom.

The notebook automatically: detects each row's case type → injects **only** the matching case-type definitions into the Groq prompt → annotates one judgment per API request → validates/parses model output robustly → checkpoints progress → skips bad rows without crashing → resumes after interruption → saves the final annotated CSV safely (the original input file is **never** modified).

## Safe testing sequence (recommended before a full run)

| Step | Settings | Purpose |
|------|----------|---------|
| 1 | `DRY_RUN = True`, `MAX_ROWS = 1` | Verify prompt construction — **no API calls, no data changes** |
| 2 | `DRY_RUN = False`, `MAX_ROWS = 5` | Annotate 5 rows for real; inspect output |
| 3 | `DRY_RUN = False`, `MAX_ROWS = None` | Full batch run |

Interrupt any time (Kernel → Interrupt / Ctrl-C): progress is checkpointed and the next run resumes automatically, skipping already-successful rows.

## Files produced (for input `criminal_batch_001.csv`)

- `criminal_batch_001_annotated.csv` — final clean output
- `criminal_batch_001_checkpoint.csv` + `_checkpoint_meta.json` — resumable progress (includes operational metadata)
- `criminal_batch_001_errors.csv` — failed rows with error details
- `criminal_batch_001_duplicates.csv` — duplicate case_id / duplicate judgment report
- `criminal_batch_001_run.log` — full run log


In [1]:
# CELL 2: Package installation
# (Safe to re-run; -q keeps output minimal.)
!pip install -q groq pandas tqdm python-dotenv


    torch (>=1.7.*)
           ~~~~~~^


In [5]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


Mon Jul  6 14:58:08 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090         On | 00000000:3B:00.0 Off |                  N/A |
|  0%   54C    P8               26W / 350W|      8MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
!kill 885 19432 885 19432

/bin/bash: line 0: kill: (885) - No such process
/bin/bash: line 0: kill: (19432) - No such process
/bin/bash: line 0: kill: (885) - No such process
/bin/bash: line 0: kill: (19432) - No such process


In [1]:
# CELL 3: Imports
import os
import re
import ast
import json
import math
import time
import random
import shutil
import hashlib
import logging
import traceback
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm

print("Imports OK. pandas", pd.__version__)


/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Imports OK. pandas 2.3.3


In [113]:
# CELL 4: USER CONFIGURATION
# ============================================================
# NORMAL OPERATION: change ONLY these two values.
# ============================================================
INPUT_FILE = "batch_data/Criminal/criminal_batch_002.csv"
OUTPUT_DIR = "final/Criminal"

# ============================================================
# ADVANCED SETTINGS (defaults are safe; change only if needed)
# ============================================================

# Groq model. ALWAYS check https://console.groq.com/docs/models for currently
# supported production models before a large run - Groq deprecates models
# regularly (e.g. llama-3.3-70b-versatile was deprecated in June 2026).
MODEL_NAME = "openai/gpt-oss-120b"

# "fill_missing"  -> only fill empty annotation fields, never overwrite existing values (default)
# "overwrite"     -> regenerate all six annotation fields for every eligible row
ANNOTATION_MODE = "fill_missing"

MAX_RETRIES = 4                 # API/parse retries per row (exponential backoff with jitter)
REQUEST_TIMEOUT = 120           # seconds per Groq request
SAVE_EVERY = 10                 # checkpoint after this many attempted rows
SLEEP_BETWEEN_REQUESTS = 1.0    # polite pause between API calls (seconds)
MAX_JUDGMENT_CHARS = 12000      # judgments longer than this are excerpted (head/middle/tail)
TEMPERATURE = 0.0               # deterministic annotation
MAX_COMPLETION_TOKENS = 2048    # cap on model output size

RETRY_FAILED_ROWS = True        # on resume, retry rows that previously failed
REUSE_DUPLICATE_ANNOTATIONS = True   # reuse annotations for byte-identical (normalized) judgments
KEEP_METADATA_IN_FINAL = False  # if True, internal _annotation_* columns are kept in the final CSV
DEBUG = False                   # if True, print raw model responses (verbose!)

# ============================================================
# TESTING CONTROLS
# ============================================================
DRY_RUN = False    # True -> load + validate + build & print one example prompt; NO API calls, NO data changes
MAX_ROWS = 20    # e.g. 5 -> annotate only the first 5 rows that need an API call; None -> all rows

# --- configuration sanity checks ---
assert ANNOTATION_MODE in ("fill_missing", "overwrite"), "ANNOTATION_MODE must be 'fill_missing' or 'overwrite'"
assert isinstance(MAX_RETRIES, int) and MAX_RETRIES >= 1, "MAX_RETRIES must be >= 1"
assert isinstance(SAVE_EVERY, int) and SAVE_EVERY >= 1, "SAVE_EVERY must be >= 1"
assert MAX_JUDGMENT_CHARS >= 2000, "MAX_JUDGMENT_CHARS is too small to be useful"
assert MAX_ROWS is None or (isinstance(MAX_ROWS, int) and MAX_ROWS >= 1), "MAX_ROWS must be None or a positive int"
print("Configuration loaded.")
print("  INPUT_FILE :", INPUT_FILE)
print("  OUTPUT_DIR :", OUTPUT_DIR)
print("  MODE       :", ANNOTATION_MODE, "| DRY_RUN:", DRY_RUN, "| MAX_ROWS:", MAX_ROWS)


Configuration loaded.
  INPUT_FILE : batch_data/Criminal/criminal_batch_002.csv
  OUTPUT_DIR : final/Criminal
  MODE       : fill_missing | DRY_RUN: False | MAX_ROWS: 20


In [114]:
# CELL 5: Logging setup and output paths
INPUT_PATH = Path(INPUT_FILE).expanduser()
OUTPUT_DIR_PATH = Path(OUTPUT_DIR).expanduser()
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

BATCH_STEM = INPUT_PATH.stem
ANNOTATED_PATH       = OUTPUT_DIR_PATH / (BATCH_STEM + "_annotated.csv")
CHECKPOINT_PATH      = OUTPUT_DIR_PATH / (BATCH_STEM + "_checkpoint.csv")
CHECKPOINT_META_PATH = OUTPUT_DIR_PATH / (BATCH_STEM + "_checkpoint_meta.json")
ERRORS_PATH          = OUTPUT_DIR_PATH / (BATCH_STEM + "_errors.csv")
DUPLICATES_PATH      = OUTPUT_DIR_PATH / (BATCH_STEM + "_duplicates.csv")
LOG_PATH             = OUTPUT_DIR_PATH / (BATCH_STEM + "_run.log")

# Never overwrite the original input CSV.
_input_resolved = INPUT_PATH.resolve()
for _p in (ANNOTATED_PATH, CHECKPOINT_PATH, ERRORS_PATH, DUPLICATES_PATH):
    if _p.resolve() == _input_resolved:
        raise RuntimeError("Output path collides with INPUT_FILE - refusing to run: " + str(_p))

logger = logging.getLogger("indilex")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False

_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
_fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
_fh.setLevel(logging.INFO)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setLevel(logging.WARNING)   # keep notebook output clean; full detail goes to the log file
_sh.setFormatter(_fmt)
logger.addHandler(_sh)

logger.info("=" * 70)
logger.info("RUN START | input=%s | output_dir=%s | model=%s | mode=%s | dry_run=%s | max_rows=%s",
            INPUT_PATH, OUTPUT_DIR_PATH, MODEL_NAME, ANNOTATION_MODE, DRY_RUN, MAX_ROWS)
print("Logging to:", LOG_PATH)


Logging to: final/Criminal/criminal_batch_002_run.log


In [115]:
# CELL 6: Groq client initialization
from dotenv import load_dotenv
load_dotenv()

# Replace CELL 6 lines 5-6 with your actual key string directly:
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
client = None

if DRY_RUN:
    print("DRY_RUN = True -> Groq client NOT initialized. No API calls will be made.")
    logger.info("DRY_RUN mode: Groq client not initialized.")
else:
    if not GROQ_API_KEY:
        raise RuntimeError(
            "GROQ_API_KEY not found. Put GROQ_API_KEY=... in a .env file next to this notebook "
            "or export it in your environment, or set DRY_RUN = True to test without the API."
        )
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY, timeout=REQUEST_TIMEOUT)
    logger.info("Groq client initialized. model=%s timeout=%ss", MODEL_NAME, REQUEST_TIMEOUT)
    print("Groq client ready. Model:", MODEL_NAME)


Groq client ready. Model: openai/gpt-oss-120b


In [116]:
# CELL 7: General helper functions

MISSING_TOKENS = {"", "nan", "none", "null"}

def is_missing(value):
    """True for None, NaN, empty/whitespace-only strings, and 'nan'/'none'/'null' (case-insensitive)."""
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower() in MISSING_TOKENS

def safe_str(value):
    """String form of a value; '' for None/NaN."""
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return str(value)

def _to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except (TypeError, ValueError):
        return default

def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

CANONICAL_CASE_TYPES = ("Criminal", "Civil", "Constitutional", "Administrative")

def normalize_case_type(value):
    """Map messy case-type strings ('  CRIMINAL ', 'civil', 'Constitutional Law', 'admin')
    to one of CANONICAL_CASE_TYPES, or None if it cannot be safely determined."""
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    s = re.sub(r"[^a-z]", "", str(value).strip().lower())
    if not s:
        return None
    if s.startswith("crim"):
        return "Criminal"
    if s.startswith("civ"):
        return "Civil"
    if s.startswith("consti"):
        return "Constitutional"
    if s.startswith("admin"):
        return "Administrative"
    return None

def normalize_judgment_for_dedup(text):
    """Whitespace-collapsed, lowercased judgment text used ONLY for exact-duplicate detection."""
    return re.sub(r"\s+", " ", safe_str(text)).strip().lower()

def judgment_hash(text):
    norm = normalize_judgment_for_dedup(text)
    if not norm:
        return ""
    return hashlib.md5(norm.encode("utf-8")).hexdigest()

def atomic_save_csv(dataframe, path):
    """Atomic CSV save: write temp file, flush, then os.replace over the target.
    UTF-8-SIG preserves Hindi/Bengali/Assamese compatibility with spreadsheet apps."""
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    dataframe.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)

def atomic_write_text(path, text):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def truncate_for_display(s, n=1500):
    s = safe_str(s)
    return s if len(s) <= n else s[:n] + "\n...[truncated for display: " + str(len(s) - n) + " more chars]..."

print("Helper functions defined.")


Helper functions defined.


In [117]:
# CELL 8: Universal annotation principles (injected for EVERY case type)
UNIVERSAL_RULES = """UNIVERSAL ANNOTATION PRINCIPLES (APPLY TO ALL CASE TYPES)

1. UNDERLYING LEGAL EVENT vs PROCEDURAL POSTURE
Every judgment may contain two layers:
(A) Procedural posture - e.g. bail application, anticipatory bail application, appeal (criminal or civil), revision, review petition, writ petition, quashing petition / application under Section 482 CrPC, injunction application, judicial review proceeding.
(B) Underlying legal event - the actual criminal offence, civil dispute, constitutional violation, or administrative action that caused the litigation.
The annotation MUST primarily represent the UNDERLYING LEGAL EVENT. Never annotate the procedural posture as the legal event.
Examples:
- Bail application arising from an alleged murder: WRONG objective_aspect = "Application for bail". CORRECT: annotate the underlying alleged murder event.
- Appeal against a decree for specific performance: WRONG = "Filing of first appeal". CORRECT: annotate the underlying contractual transaction and alleged breach.
- Writ petition challenging detention: do not annotate merely "Filing of writ petition"; annotate the challenged detention action and the alleged constitutional violation.
- Judicial review of a licence cancellation: annotate the licence cancellation as the impugned action and the supported grounds for review.

2. THE JUDGMENT TEXT IS THE SOLE SOURCE OF TRUTH
Do NOT use external knowledge. Do NOT invent party names, victim names, accused names, facts, motives, mens rea, legal provisions, statutory section numbers, remedies, constitutional violations, or grounds of judicial review. Do NOT assume facts merely because they are legally plausible. Every annotation must be grounded in the supplied judgment text.

3. ALLEGATION vs JUDICIAL FINDING
Carefully distinguish: allegation, prosecution case, complainant statement, plaintiff claim, petitioner contention, respondent argument, evidence, judicial observation, judicial finding. Use legally careful wording such as: "The prosecution alleged that...", "The plaintiff claimed that...", "The petitioner contended that...", "The judgment records that...", "The court found that...". Never present an allegation as an established judicial fact.

4. OCR AND LOW-QUALITY TEXT
Indian court judgments may contain OCR errors, corrupted names, malformed punctuation, broken section numbers, encoding errors, incomplete sentences, and scanning artifacts. Handle these conservatively: do not invent names from corrupted text; do not create statutory sections from garbled numbers; use surrounding context only when an interpretation is reasonably supported; if uncertain, prefer a conservative annotation. Uncertainty is better than fabrication."""

print("UNIVERSAL_RULES loaded:", len(UNIVERSAL_RULES), "chars")


UNIVERSAL_RULES loaded: 2700 chars


In [118]:
# CELL 9: Criminal case definitions (injected ONLY for Criminal rows)
CRIMINAL_DEFINITIONS = """CASE-TYPE DEFINITIONS: CRIMINAL

1. SUBJECT - ACCUSED
The Subject is the accused person, alleged offender, convicted person, acquitted person whose conduct is under appellate review, abettor, conspirator, or other legally responsible actor connected to the underlying criminal event. It answers: "Who allegedly committed, participated in, abetted, conspired in, or is legally responsible for the criminal act?"
The Subject may include: named accused in the FIR; accused in the charge sheet; convicted person; acquitted accused under appellate review; abettor; conspirator; corporate entity accused of criminal conduct; public servant accused under corruption law; multiple accused materially connected to one criminal event.
Identification rules:
1. Use specific names when clearly identifiable.
2. Do NOT automatically treat the Applicant as Subject.
3. Do NOT automatically treat the Petitioner as Subject.
4. Do NOT automatically treat the Appellant as Subject.
5. Determine the person's substantive role in the underlying crime.
6. Do not include advocates. 7. Do not include judges. 8. Do not include witnesses merely because they testified. 9. Do not include government officials appearing only procedurally.
10. For multiple accused, include the materially relevant accused persons.
11. Never invent names from OCR-corrupted text.
Common mistakes:
- WRONG: Subject = "Applicant" (the applicant may be an informant or another procedural party).
- WRONG: Subject = "Appellant" (in a State appeal against acquittal, the appellant is the State, while the underlying accused remain the criminal Subject).
- WRONG: Subject = an advocate's name (advocates are representatives, not parties to the criminal event).

2. OBJECT - VICTIM
The Object is the victim, injured person, deceased person, affected person, affected property interest, or legally protected interest against which the alleged criminal conduct was directed. It answers: "Who or what was directly affected, harmed, or targeted by the alleged crime?"
Possible Objects: individual victim; injured person; deceased person; complainant when the complainant is also the victim; wife affected by matrimonial cruelty; child affected by the offence; property owner; affected property interest; multiple victims; State/Society only for genuinely collective offences.
Identification rules:
1. Prefer the actual victim over the generic label "Complainant".
2. If the FIR informant and the victim are different, identify the actual victim.
3. In homicide cases, the deceased is the primary Object.
4. Do not automatically use State/Society.
5. Do not use the procedural opposite party as Object.
6. Use specific names where clearly supported.
Common mistakes:
- WRONG: Object = FIR informant, when the FIR was filed by the victim's father in a murder case. CORRECT: Object = the deceased victim.
- WRONG: Object = "State" in an ordinary personal assault case with an identifiable injured victim.

3. OBJECTIVE ASPECT - CRIMINAL ACT
The Objective Aspect is the externally observable criminal act, conduct, omission, transaction, or legally relevant behaviour alleged in the underlying criminal event. It answers: "What did the accused allegedly do?"
Examples of relevant conduct: assault; stabbing; shooting; strangulation; beating; theft; breaking locks; arson; property damage; cheating; fraudulent representation; forgery; matrimonial cruelty; dowry harassment; sexual offence; conspiracy; abetment; illegal possession; trafficking; legally relevant omission.
Rules:
1. Describe the actual alleged conduct. WRONG: "Offence under Section 302 IPC". BETTER: "Alleged killing of the deceased by stabbing with a knife, according to the prosecution case."
2. Do not merely list statutory provisions.
3. Keep the annotation concise but factually meaningful.
4. Preserve relevant details where available: weapon, method, manner of injury, fraudulent technique, collective action, transaction.
5. Do not treat bail, appeal, revision, or quashing as the criminal act.
6. For connected offences forming one transaction, describe a coherent event.
7. Distinguish allegation from judicial finding.

4. SUBJECTIVE ASPECT - CRIMINAL INTENT / MENS REA
The Subjective Aspect is the legally relevant mental state associated with the alleged criminal conduct, as supported by the judgment. It answers: "What was the legally relevant intention, knowledge, motive, or purpose associated with the alleged act?"
Possible supported mental states: intention; knowledge; dishonest intention; fraudulent intention; common intention; conspiracy purpose; dowry-related motive; intention to insult; intention to intimidate; intention to cause alarm; knowledge of likely death; motive such as revenge, greed, enmity, or property dispute.
Rules:
1. Never invent mens rea.
2. Do not infer intention only from a section number.
3. Do not assume Section 302 automatically proves intent to kill.
4. Do not assume Section 498-A automatically means dowry motive.
5. A mental state may be annotated only where it is: explicitly discussed; clearly supported by facts; supported by motive evidence; necessarily supported by strongly described conduct; or explicitly part of the judicial analysis.
6. Avoid vague labels such as "criminal intention" or "mens rea" without specifying the actual supported mental element.
7. When multiple mental states are supported, include them.
8. If the mental state genuinely cannot be determined, return exactly: "Not Clearly Specified".
Common mistakes:
- WRONG: "Dowry-related motive" when the judgment mentions cruelty but no dowry demand. CORRECT: "Not Clearly Specified".
- WRONG: "Intent to kill" merely because Section 307 or Section 302 is mentioned. The mental element must be evaluated from the judgment context."""

print("CRIMINAL_DEFINITIONS loaded:", len(CRIMINAL_DEFINITIONS), "chars")


CRIMINAL_DEFINITIONS loaded: 5772 chars


In [119]:
# CELL 10: Civil case definitions (injected ONLY for Civil rows)
CIVIL_DEFINITIONS = """CASE-TYPE DEFINITIONS: CIVIL

1. SUBJECT - DEFENDANT
The Subject is the defendant, respondent, or substantive party against whom civil relief is sought in the underlying dispute. It answers: "Against whom is the civil claim directed?" and "Who allegedly committed the civil wrong, breached the obligation, or caused the actionable injury?"
Possible Subjects: named defendant; party who breached the contract; alleged tortfeasor; encroaching party; judgment debtor; corporate entity sued for civil liability; government body sued in its civil capacity; landlord or tenant depending on the substantive claim; bank or financial institution where relevant.
Rules:
1. Identify the substantive role in the original dispute.
2. Do not confuse the appeal position with the original legal role.
3. The original defendant remains the Subject even if now the appellant.
4. Do not treat every respondent as the substantive defendant.
5. Use specific names when supported.
Common mistake: in an appeal filed by the original defendant - WRONG: Subject = the current respondent simply because of the appeal caption. CORRECT: Subject = the substantive defendant from the underlying civil dispute.

2. OBJECT - PLAINTIFF / CLAIMANT
The Object is the plaintiff, claimant, consumer, decree holder, or person whose civil right or interest was allegedly injured and who seeks a remedy. It answers: "Who suffered the civil injury, loss, deprivation, or interference with rights?"
Rules:
1. Prefer the actual injured civil party.
2. Do not rely only on the appellate procedural position.
3. In property disputes, identify the affected party and property interest where relevant.
4. In consumer disputes, identify the affected consumer.
5. In insurance disputes, identify the insured or beneficiary where supported.
6. Use specific names where available.

3. OBJECTIVE ASPECT - CAUSE OF ACTION
The Objective Aspect is the cause of action: the facts, transactions, acts, or omissions that give rise to the plaintiff's right to seek a judicial remedy. It answers: "What happened?" and "What did the defendant allegedly do or fail to do?"
Possible categories: breach of contract; property dispute; encroachment; disputed title; partition; negligence; nuisance; defamation; matrimonial dispute; recovery claim; specific performance dispute; tenancy dispute; consumer dispute; succession dispute; intellectual property dispute; arbitration-related dispute.
Rules:
1. Describe the factual cause of action, not only the legal category. WRONG: "Breach of contract". BETTER: "Alleged refusal of the seller to execute the agreed sale deed despite receipt of consideration under the agreement."
2. Include legally relevant facts when available.
3. Do not describe procedural history as the cause of action.
4. In appeals, annotate the original underlying dispute.

4. SUBJECTIVE ASPECT - REMEDY SOUGHT
The Subjective Aspect in a Civil case is the specific judicial remedy, order, or outcome sought by the claimant. It answers: "What does the aggrieved civil party want the court to do?"
Possible remedies: specific performance; permanent injunction; mandatory injunction; temporary injunction; damages; compensation; declaration; recovery of money; recovery of possession; partition; eviction; restitution; rendering of accounts; cancellation of deed; rectification; refund; replacement; consumer compensation.
Rules:
1. Be specific. 2. Do not output the generic word "Relief". 3. Include multiple material remedies when supported. 4. Distinguish the original remedy from appellate relief where relevant."""

print("CIVIL_DEFINITIONS loaded:", len(CIVIL_DEFINITIONS), "chars")


CIVIL_DEFINITIONS loaded: 3569 chars


In [120]:
# CELL 11: Constitutional case definitions (injected ONLY for Constitutional rows)
CONSTITUTIONAL_DEFINITIONS = """CASE-TYPE DEFINITIONS: CONSTITUTIONAL

1. SUBJECT - STATE RESPONDENT / PUBLIC AUTHORITY
The Subject is the State respondent, government authority, statutory body, public official, legislature, or entity exercising public power whose action, inaction, law, or policy is constitutionally challenged. It answers: "Whose action, law, policy, or conduct is challenged as constitutionally impermissible?"
Possible Subjects: Union Government; State Government; ministry; government department; constitutional body; statutory body; local authority; public official; legislature; Article 12 instrumentality; a public-function entity where legally relevant.
Rules:
1. Identify the actual authority whose conduct is challenged.
2. Do not simply copy all respondents.
3. If a statute is challenged, identify the responsible legislative/government authority.
4. If an executive order is challenged, identify the issuing authority.
5. In detention matters, identify the detaining authority.

2. OBJECT - PETITIONER / ACTUAL RIGHTS HOLDER
The Object is the person, group, class, or rights-holder whose constitutional rights or entitlements are allegedly violated or threatened.
Rules:
1. Distinguish the procedural petitioner from the actual affected person.
2. In PIL, identify the actual affected public or class where appropriate.
3. In habeas corpus, the detained person is the Object even if a relative filed the petition.
4. Use specific names where supported.

3. OBJECTIVE ASPECT - CONSTITUTIONAL BREACH
The Objective Aspect is the alleged constitutional violation, fundamental-right infringement, or unconstitutional State action forming the basis of the petition. It answers: "What State action allegedly violated which constitutional right?"
Possible areas: Article 14 equality violation; arbitrary classification; discriminatory treatment; Article 19 freedom restriction; Article 21 life and liberty violation; illegal detention; privacy violation; livelihood issue; Article 21-A education rights; religious freedom; unconstitutional service action; legislative invalidity; executive overreach; electoral constitutional issues.
Rules:
1. Identify the constitutional provision when supported.
2. Describe the challenged State action.
3. Connect the action with the alleged constitutional violation.
4. Do not output only "Violation of Article 14" if a factual basis is available.

4. SUBJECTIVE ASPECT - WRIT RELIEF
The Subjective Aspect is the constitutional or writ remedy sought.
Possible remedies: habeas corpus; mandamus; certiorari; prohibition; quo warranto; declaration of unconstitutionality; constitutional direction; compensation; stay or interim relief.
Rules:
1. Specify the type of writ where identifiable.
2. Describe the practical relief sought.
3. Include relevant interim relief.
4. In habeas corpus, identify production/release relief where supported."""

print("CONSTITUTIONAL_DEFINITIONS loaded:", len(CONSTITUTIONAL_DEFINITIONS), "chars")


CONSTITUTIONAL_DEFINITIONS loaded: 2866 chars


In [121]:
# CELL 12: Administrative case definitions (injected ONLY for Administrative rows)
ADMINISTRATIVE_DEFINITIONS = """CASE-TYPE DEFINITIONS: ADMINISTRATIVE

1. SUBJECT - ADMINISTRATIVE AUTHORITY
The Subject is the administrative authority, government body, regulator, statutory authority, quasi-judicial body, or public official whose administrative action is challenged. It answers: "Which administrative body or official made the challenged decision?"
Possible Subjects: government department; regulator; statutory authority; tribunal; municipal authority; revenue authority; tax authority; educational authority; licensing authority; disciplinary authority; tender authority.
Rules:
1. Identify the specific authority that made the decision.
2. Do not merely copy all procedural respondents.
3. Where possible identify both the department and the responsible officer.
4. Distinguish the policy-maker from the implementer.
5. Annotate the authority whose actual action is under challenge.

2. OBJECT - AGGRIEVED PARTY
The Object is the person, entity, or class directly and adversely affected by the administrative action.
Possible Objects: citizen; government employee; business entity; landowner; contractor; bidder; student; candidate; pensioner; taxpayer; affected community.
Rules:
1. Prefer a specific name or description.
2. In representative matters, identify the affected class.
3. Identify the actually adversely affected party, not merely the procedural petitioner.

3. OBJECTIVE ASPECT - IMPUGNED ACTION
The Objective Aspect is the specific administrative decision, order, notification, rule, policy, omission, or administrative conduct challenged. It answers: "What exactly did the authority do or fail to do?"
Examples: transfer order; termination order; suspension order; demolition order; eviction order; tender award; selection or rejection decision; disciplinary action; licence grant, refusal, or cancellation; revenue action; tax assessment; penalty; demand notice; denial of promotion; blacklisting; regulatory direction; environmental clearance; failure to decide an application.
Where available include: date; order number; issuing authority; nature of the action; effect on the aggrieved party.
Common mistake - WRONG: "Administrative action". BETTER: "Order dated [date], issued by [authority], cancelling the petitioner's licence for the reasons recorded in the judgment."

4. SUBJECTIVE ASPECT - GROUNDS FOR JUDICIAL REVIEW
The Subjective Aspect is the legal basis or administrative-law ground on which the challenged action is alleged to be invalid. It answers: "Why is the administrative action legally challenged?"
Possible grounds: illegality; lack of jurisdiction; ultra vires action; irrationality; Wednesbury unreasonableness; procedural impropriety; violation of natural justice; lack of hearing; lack of notice; bias; proportionality; legitimate expectation; arbitrariness; mala fide action; statutory violation; discrimination; error apparent on the face of the record; failure to consider relevant factors; consideration of irrelevant factors; fettering of discretion; promissory estoppel.
Rules:
1. Use only grounds supported by the judgment.
2. Connect each ground to its factual basis.
3. Distinguish grounds argued from grounds accepted.
4. Include multiple material grounds where supported.
WRONG: "Violation of natural justice". BETTER: "Alleged violation of natural justice because the cancellation order was passed without prior notice or opportunity of hearing." """

print("ADMINISTRATIVE_DEFINITIONS loaded:", len(ADMINISTRATIVE_DEFINITIONS), "chars")


ADMINISTRATIVE_DEFINITIONS loaded: 3394 chars


In [122]:
# CELL 13: Legal provision definition (injected for EVERY case type)
LEGAL_PROVISION_RULES = """LEGAL PROVISION FIELD (legal_provision) - ALL CASE TYPES

The legal_provision field must contain statutory sections, constitutional articles, Acts, Rules, Regulations, or other legal provisions that are actually mentioned or clearly applied in the judgment.
Rules:
1. Never invent provisions.
2. Preserve section and article numbers accurately.
3. Include the statute name where identifiable.
4. Deduplicate repeated provisions.
5. Separate multiple provisions using semicolons.
6. Prioritize substantive provisions governing the underlying legal event.
7. Include procedural provisions where relevant to the legal context.
8. Handle OCR-corrupted numbers conservatively.
9. Never create a provision from uncertain garbled text.
Important Criminal-law distinction: Section 482 -> CrPC; Section 438 -> CrPC; Section 439 -> CrPC; Section 437 -> CrPC; Section 164 -> CrPC; Section 156(3) -> CrPC; Section 82 -> CrPC. Do not incorrectly label these as IPC.
Possible special statutes (only where actually supported by the text): SC/ST (Prevention of Atrocities) Act; Dowry Prohibition Act; POCSO Act; NDPS Act; Prevention of Corruption Act; Arms Act; Information Technology Act.
Output format example: "Section 302 IPC; Section 34 IPC; Section 439 CrPC"
If no provision can be reliably extracted, return "" (empty string)."""

print("LEGAL_PROVISION_RULES loaded:", len(LEGAL_PROVISION_RULES), "chars")


LEGAL_PROVISION_RULES loaded: 1317 chars


In [123]:
# CELL 14: Reasoning definition (injected for EVERY case type)
REASONING_RULES = """REASONING FIELD (reasoning) - ALL CASE TYPES

The reasoning field is NOT a generic summary of the judgment. It is a concise, evidence-grounded explanation of WHY the final annotation fields are supported by the judgment text - the connecting justification between party roles, the underlying legal event, the Objective Aspect, the Subjective Aspect, the legal provisions, and the procedural context.
Target length: 50-100 words.
Where applicable, the reasoning should address:
1. why the identified Subject has that substantive role;
2. why the identified Object is the affected party or interest;
3. what underlying event supports the Objective Aspect;
4. what evidence supports the Subjective Aspect;
5. which legal provisions apply;
6. whether relevant facts are allegations or findings;
7. how the procedural posture differs from the underlying legal event.
The reasoning must NOT: become a general judgment summary; repeat the full judgment; invent facts; invent legal conclusions; use unrelated boilerplate; contradict the extracted annotation fields; falsely present allegations as findings.
Use legally careful language: "The prosecution alleged that...", "The plaintiff claimed that...", "The petitioner contended that...", "The judgment records that...", "The court found that..." """

print("REASONING_RULES loaded:", len(REASONING_RULES), "chars")


REASONING_RULES loaded: 1291 chars


In [124]:
# CELL 15: Case-type normalization and conditional definition selector
CASE_DEFINITIONS = {
    "Criminal": CRIMINAL_DEFINITIONS,
    "Civil": CIVIL_DEFINITIONS,
    "Constitutional": CONSTITUTIONAL_DEFINITIONS,
    "Administrative": ADMINISTRATIVE_DEFINITIONS,
}

def get_case_definition(case_type):
    """Return (canonical_case_type, definitions_text) for a (possibly messy) case-type value.
    Raises ValueError if the case type cannot be safely determined."""
    ct = normalize_case_type(case_type)
    if ct is None:
        raise ValueError("Unsupported or undetectable case_type: " + repr(case_type))
    return ct, CASE_DEFINITIONS[ct]

def detect_row_case_type(raw_value, file_dominant=None, folder_hint=None, filename_hint=None):
    """Fallback chain for case-type detection:
    1. the row's own case_type value
    2. the dominant case type of the whole file
    3. the parent folder name
    4. the input filename
    Returns a canonical case type, or None if it cannot be safely determined."""
    for candidate in (raw_value, file_dominant, folder_hint, filename_hint):
        ct = normalize_case_type(candidate)
        if ct is not None:
            return ct
    return None

print("Case definition selector ready. Supported:", ", ".join(CASE_DEFINITIONS))


Case definition selector ready. Supported: Criminal, Civil, Constitutional, Administrative


In [125]:
# CELL 16: System prompt builder (conditional definition injection)
OUTPUT_RULES = """OUTPUT FORMAT (STRICT)

Return ONLY a single JSON object with exactly these keys:
{"subject": "", "object": "", "objective_aspect": "", "subjective_aspect": "", "legal_provision": "", "reasoning": ""}
No Markdown. No code fences. No introductory explanation. No text after the JSON.
All values must be strings grounded strictly in the supplied judgment text.
If the subjective aspect genuinely cannot be determined in a Criminal case, use exactly "Not Clearly Specified".
If no legal provision can be reliably extracted, use "" for legal_provision.
Prefer conservative, shorter values over fabricated detail."""

SYSTEM_PREAMBLE = ("You are an expert Indian legal annotation system. You annotate one court judgment at a time "
                   "with legally precise, conservative, text-grounded values. Follow every rule below exactly.")

def build_system_prompt(case_type):
    """Build the system prompt for one canonical case type.
    Includes ONLY: universal principles + the detected case type's definitions +
    legal-provision rules + reasoning rules + output rules.
    Definitions for the other three case types are NEVER included."""
    ct, definitions = get_case_definition(case_type)
    parts = [SYSTEM_PREAMBLE, UNIVERSAL_RULES, definitions, LEGAL_PROVISION_RULES, REASONING_RULES, OUTPUT_RULES]
    return "\n\n" .join(parts)

# quick sanity check that conditional injection works (no API involved)
_sp = build_system_prompt("criminal")
assert "CASE-TYPE DEFINITIONS: CRIMINAL" in _sp
assert "CASE-TYPE DEFINITIONS: CIVIL" not in _sp
assert "CASE-TYPE DEFINITIONS: CONSTITUTIONAL" not in _sp
assert "CASE-TYPE DEFINITIONS: ADMINISTRATIVE" not in _sp
del _sp
print("System prompt builder ready (conditional injection verified).")


System prompt builder ready (conditional injection verified).


In [126]:
# CELL 17: User prompt builder
def build_user_prompt(case_id, language, case_type, fields_to_generate, judgment_text):
    """Build the per-row user prompt. One judgment per request."""
    return (
        "ANNOTATION REQUEST\n"
        "case_id: " + safe_str(case_id) + "\n"
        "language: " + safe_str(language) + "\n"
        "case_type: " + safe_str(case_type) + "\n"
        "fields_to_generate: " + ", ".join(fields_to_generate) + "\n\n"
        "Return the complete six-key JSON object described in the output rules. "
        "The fields listed in fields_to_generate are the ones that will be used; "
        "existing annotations for other fields are preserved and your values for them will be ignored.\n\n"
        "JUDGMENT TEXT (sole and final source of truth):\n"
        "<<<JUDGMENT_START>>>\n"
        + safe_str(judgment_text) + "\n"
        "<<<JUDGMENT_END>>>\n\n"
        "Return ONLY the JSON object."
    )

print("User prompt builder ready.")


User prompt builder ready.


In [127]:
# CELL 18: Long judgment handler
# Strategy (documented here and logged whenever applied):
# If a judgment exceeds MAX_JUDGMENT_CHARS, we send three clearly separated excerpts
# from the SAME judgment instead of blindly truncating:
#   - HEAD  (~45% of budget): case caption, parties, FIR/plaint/petition facts
#   - MIDDLE (~20% of budget): evidence / arguments region around the document centre
#   - TAIL  (~35% of budget): the court's reasoning and final order
# Explicit markers tell the model that material was omitted, so it stays conservative.
# This keeps every request safely under token limits without losing the legally
# densest regions (beginning facts + final reasoning/order).

TRUNCATION_NOTE = ("[NOTE TO ANNOTATOR MODEL: This judgment exceeds the length limit. "
                   "The excerpts below are the beginning, a middle portion, and the final portion "
                   "of the SAME judgment. Material between excerpts has been omitted. "
                   "Annotate conservatively from what is present; do not guess omitted content.]")
OMISSION_MARKER = "\n\n[... PORTION OF THE JUDGMENT OMITTED DUE TO LENGTH ...]\n\n"

def prepare_judgment_text(text, max_chars=None):
    """Return (prompt_ready_text, was_truncated). Never raises on odd input."""
    if max_chars is None:
        max_chars = MAX_JUDGMENT_CHARS
    t = safe_str(text)
    if len(t) <= max_chars:
        return t, False
    overhead = len(TRUNCATION_NOTE) + 2 * len(OMISSION_MARKER) + 8
    budget = max(max_chars - overhead, 1000)
    head_n = int(budget * 0.45)
    tail_n = int(budget * 0.35)
    mid_n = budget - head_n - tail_n
    head = t[:head_n]
    tail = t[-tail_n:]
    centre = len(t) // 2
    mid = t[centre - mid_n // 2 : centre + mid_n - mid_n // 2] if mid_n > 200 else ""
    if mid:
        excerpt = TRUNCATION_NOTE + "\n\n" + head + OMISSION_MARKER + mid + OMISSION_MARKER + tail
    else:
        excerpt = TRUNCATION_NOTE + "\n\n" + head + OMISSION_MARKER + tail
    return excerpt, True

print("Long judgment handler ready (MAX_JUDGMENT_CHARS =", MAX_JUDGMENT_CHARS, ").")


Long judgment handler ready (MAX_JUDGMENT_CHARS = 12000 ).


In [128]:
# CELL 19: Robust JSON parser and schema validator
REQUIRED_KEYS = ["subject", "object", "objective_aspect", "subjective_aspect", "legal_provision", "reasoning"]

def _balanced_json_slice(s):
    """Extract the first balanced {...} block, respecting strings/escapes.
    If the block never closes (truncated output), return from '{' to the end."""
    start = s.find("{")
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start : i + 1]
    return s[start:]  # truncated candidate

def _try_loads(candidate):
    """Try hard to turn a candidate string into a dict. Returns dict or None."""
    if not candidate:
        return None
    attempts = [candidate]
    # trailing-comma repair
    attempts.append(re.sub(r",\s*([}\]])", r"\1", candidate))
    # truncated-output repair
    stripped = candidate.rstrip()
    if not stripped.endswith("}"):
        attempts.append(stripped + '"}')
        attempts.append(stripped + "}")
        attempts.append(stripped + '"}' + "}")
    for a in attempts:
        try:
            obj = json.loads(a)
            if isinstance(obj, dict):
                return obj
        except (json.JSONDecodeError, ValueError):
            pass
    # single-quoted / Python-literal style dict (safe repair via literal_eval)
    try:
        obj = ast.literal_eval(candidate)
        if isinstance(obj, dict):
            return obj
    except (ValueError, SyntaxError, MemoryError, RecursionError, TypeError):
        pass
    return None

def robust_parse_model_output(raw):
    """Parse a model response into a validated annotation dict.
    Handles: clean JSON, ```json fences, prefix/suffix text, whitespace/newlines,
    null values, missing keys, extra keys, single-quote dicts, trailing commas,
    mildly truncated output, and empty responses.
    Returns a dict with exactly the six REQUIRED_KEYS (missing -> "", null -> "",
    extra keys ignored), or None if the response is unusable (caller retries)."""
    if raw is None:
        return None
    s = str(raw).strip()
    if not s:
        return None
    s = re.sub(r"```(?:json)?", "", s, flags=re.IGNORECASE).strip()
    candidates = []
    sliced = _balanced_json_slice(s)
    if sliced:
        candidates.append(sliced)
    i, j = s.find("{"), s.rfind("}")
    if i != -1 and j > i:
        candidates.append(s[i : j + 1])
    candidates.append(s)
    parsed = None
    for c in candidates:
        parsed = _try_loads(c)
        if parsed is not None:
            break
    if parsed is None:
        return None
    out = {}
    present = 0
    for k in REQUIRED_KEYS:
        if k in parsed:
            present += 1
        v = parsed.get(k)
        out[k] = "" if v is None else str(v).strip()
    if present == 0:
        return None  # a dict, but not our schema at all -> treat as parse failure
    return out

print("Robust JSON parser ready.")


Robust JSON parser ready.


In [129]:
# CELL 20: Groq API function with retries (exponential backoff + jitter)
API_CALL_COUNT = 0

def call_groq_annotation(system_prompt, user_prompt, case_id=""):
    """Call Groq for one judgment. Retries recoverable failures (timeouts, connection
    errors, rate limits, 5xx, malformed/empty/unparseable model output) with
    exponential backoff (~2s, 4s, 8s, 16s) plus jitter. Never retries forever.
    Returns (annotation_dict_or_None, last_error_or_None, attempts_used)."""
    global API_CALL_COUNT
    if client is None:
        raise RuntimeError("Groq client is not initialized (DRY_RUN mode or missing API key).")
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            API_CALL_COUNT += 1
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_COMPLETION_TOKENS,
            )
            raw = response.choices[0].message.content
            if DEBUG:
                print("---- RAW MODEL RESPONSE (case_id=" + safe_str(case_id) + ", attempt " + str(attempt) + ") ----")
                print(truncate_for_display(raw, 3000))
            parsed = robust_parse_model_output(raw)
            if parsed is not None:
                return parsed, None, attempt
            last_error = "unparseable_or_empty_model_output"
            logger.warning("case_id=%s attempt %d/%d: unparseable model output; will retry.",
                           safe_str(case_id), attempt, MAX_RETRIES)
        except KeyboardInterrupt:
            raise
        except Exception as exc:  # timeouts, rate limits, connection errors, 5xx, SDK errors
            last_error = (type(exc).__name__ + ": " + str(exc))[:500]
            logger.warning("case_id=%s attempt %d/%d: API error: %s",
                           safe_str(case_id), attempt, MAX_RETRIES, last_error)
        if attempt < MAX_RETRIES:
            backoff = min(2 ** attempt, 60) + random.uniform(0, 1)
            time.sleep(backoff)
    return None, last_error, MAX_RETRIES

print("Groq API caller ready (MAX_RETRIES =", MAX_RETRIES, ").")


Groq API caller ready (MAX_RETRIES = 4 ).


In [130]:
# CELL 21: CSV loading and validation
SOURCE_COLUMNS = ["case_id", "language", "case_type", "judgment_text"]   # NEVER modified
TARGET_COLUMNS = ["subject", "object", "objective_aspect", "subjective_aspect", "legal_provision", "reasoning"]
FINAL_COLUMNS = SOURCE_COLUMNS + TARGET_COLUMNS
METADATA_COLUMNS = ["_internal_row_id", "_judgment_hash", "_detected_case_type",
                    "_annotation_status", "_annotation_error", "_annotation_attempts",
                    "_annotation_timestamp", "_model_used"]
RESTORE_METADATA_COLUMNS = ["_annotation_status", "_annotation_error", "_annotation_attempts",
                            "_annotation_timestamp", "_model_used"]

def load_input_csv(path):
    """Load the batch CSV with everything as strings ('' instead of NaN) and
    encoding fallbacks for real-world files."""
    if not Path(path).is_file():
        raise FileNotFoundError("INPUT_FILE not found: " + str(path))
    last_exc = None
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding=enc)
            return frame, enc
        except UnicodeDecodeError as exc:
            last_exc = exc
    raise last_exc

df, USED_ENCODING = load_input_csv(INPUT_PATH)
df.columns = [str(c).strip() for c in df.columns]
ORIGINAL_COLUMNS = list(df.columns)

if "judgment_text" not in df.columns:
    raise ValueError("Input CSV has no 'judgment_text' column - cannot annotate. Columns found: "
                     + ", ".join(ORIGINAL_COLUMNS))

CREATED_COLUMNS = []
for col in ["case_id", "language", "case_type"]:
    if col not in df.columns:
        df[col] = ""
        CREATED_COLUMNS.append(col)
        logger.warning("Source column '%s' missing in input - created as empty.", col)
for col in TARGET_COLUMNS:
    if col not in df.columns:
        df[col] = ""
        CREATED_COLUMNS.append(col)

df = df.fillna("")

# Any original columns beyond the standard schema are preserved and re-attached
# after the standard columns in the final output.
EXTRA_ORIGINAL_COLUMNS = [c for c in ORIGINAL_COLUMNS if c not in FINAL_COLUMNS]

# --- stable internal row identity (never alters the original case_id) ---
_cid_series = df["case_id"].map(lambda v: safe_str(v).strip())
_cid_counts = _cid_series[_cid_series.map(lambda v: not is_missing(v))].value_counts().to_dict()

def _make_internal_id(pos, cid, judgment):
    cid = safe_str(cid).strip()
    if not is_missing(cid) and _cid_counts.get(cid, 0) == 1:
        return "cid::" + cid
    h = judgment_hash(judgment) or hashlib.md5(("emptyrow:" + str(pos)).encode("utf-8")).hexdigest()
    return "row::" + format(pos, "06d") + "::" + h[:12]

df["_internal_row_id"] = [
    _make_internal_id(pos, df.iloc[pos]["case_id"], df.iloc[pos]["judgment_text"])
    for pos in range(len(df))
]
df["_judgment_hash"] = df["judgment_text"].map(judgment_hash)

# --- case-type detection with fallback chain (row value -> file dominant -> folder -> filename) ---
FOLDER_HINT = INPUT_PATH.parent.name
FILENAME_HINT = BATCH_STEM
_row_types = df["case_type"].map(normalize_case_type)
_mode = _row_types.dropna().mode()
FILE_DOMINANT_CASE_TYPE = _mode.iloc[0] if len(_mode) else None

df["_detected_case_type"] = df["case_type"].map(
    lambda v: detect_row_case_type(v, FILE_DOMINANT_CASE_TYPE, FOLDER_HINT, FILENAME_HINT) or ""
)

# --- operational metadata defaults ---
df["_annotation_status"] = "pending"
df["_annotation_error"] = ""
df["_annotation_attempts"] = "0"
df["_annotation_timestamp"] = ""
df["_model_used"] = ""

# Snapshot of the source columns for the final integrity check: these must be
# byte-identical when we save the final output.
SOURCE_SNAPSHOT = df[SOURCE_COLUMNS].copy(deep=True).reset_index(drop=True)

logger.info("Loaded %d rows from %s (encoding=%s). Created columns: %s. Dominant case type: %s.",
            len(df), INPUT_PATH.name, USED_ENCODING, CREATED_COLUMNS or "none", FILE_DOMINANT_CASE_TYPE)
print("Loaded", len(df), "rows (encoding:", USED_ENCODING + ").")
print("Missing columns created:", CREATED_COLUMNS or "none")
print("Case-type hints -> file dominant:", FILE_DOMINANT_CASE_TYPE, "| folder:", FOLDER_HINT, "| filename:", FILENAME_HINT)


Loaded 20 rows (encoding: utf-8-sig).
Missing columns created: none
Case-type hints -> file dominant: Criminal | folder: Criminal | filename: criminal_batch_002


In [131]:
# CELL 22: Dataset summary and duplicate analysis (MUST run before annotation)
_nonempty_judgment = ~df["judgment_text"].map(is_missing)
_detected_ok = df["_detected_case_type"].map(lambda v: normalize_case_type(v) is not None)

def _row_annotation_state(row):
    missing_flags = [is_missing(row[f]) for f in TARGET_COLUMNS]
    if not any(missing_flags):
        return "full"
    if all(missing_flags):
        return "empty"
    return "partial"

_states = df.apply(_row_annotation_state, axis=1)
fully_annotated = int((_states == "full").sum())
partially_annotated = int((_states == "partial").sum())
empty_judgment_count = int((~_nonempty_judgment).sum())
case_type_undetectable = int((_nonempty_judgment & ~_detected_ok).sum())

if ANNOTATION_MODE == "overwrite":
    _needs_api_mask = _nonempty_judgment & _detected_ok
else:
    _needs_api_mask = _nonempty_judgment & _detected_ok & (_states != "full")
rows_requiring_api = int(_needs_api_mask.sum())

# --- duplicates (report only; never auto-deleted) ---
_dup_cid_mask = _cid_series.map(lambda v: not is_missing(v)) & _cid_series.duplicated(keep=False)
_dup_jdg_mask = (df["_judgment_hash"] != "") & df["_judgment_hash"].duplicated(keep=False)
duplicate_case_id_count = int(_dup_cid_mask.sum())
duplicate_judgment_count = int(_dup_jdg_mask.sum())

DUPLICATES_REPORT_DF = df.loc[_dup_cid_mask | _dup_jdg_mask,
                              ["_internal_row_id", "case_id", "_judgment_hash"]].copy()
DUPLICATES_REPORT_DF["duplicate_case_id"] = _dup_cid_mask[_dup_cid_mask | _dup_jdg_mask]
DUPLICATES_REPORT_DF["duplicate_judgment_text"] = _dup_jdg_mask[_dup_cid_mask | _dup_jdg_mask]

print("=" * 68)
print("INPUT VALIDATION SUMMARY")
print("=" * 68)
print("Input filename              :", INPUT_PATH.name)
print("Output directory            :", OUTPUT_DIR_PATH)
print("Total rows                  :", len(df))
print("Columns found               :", ", ".join(ORIGINAL_COLUMNS))
print("Missing columns created     :", ", ".join(CREATED_COLUMNS) if CREATED_COLUMNS else "none")
print("Case type distribution (raw):", dict(df["case_type"].replace("", "<blank>").value_counts()))
print("Case type (detected)        :", dict(df["_detected_case_type"].replace("", "<undetectable>").value_counts()))
print("Language distribution       :", dict(df["language"].replace("", "<blank>").value_counts()))
print("Empty judgment_text rows    :", empty_judgment_count)
print("Fully annotated rows        :", fully_annotated)
print("Partially annotated rows    :", partially_annotated)
print("Rows requiring API annotation:", rows_requiring_api, "(mode:", ANNOTATION_MODE + ")")
print("Case type undetectable rows :", case_type_undetectable)
print("Duplicate case_id rows      :", duplicate_case_id_count)
print("Exact duplicate judgments   :", duplicate_judgment_count)
print("=" * 68)
logger.info("Validation summary: rows=%d api_needed=%d empty_judgment=%d full=%d partial=%d dup_cid=%d dup_jdg=%d undetectable=%d",
            len(df), rows_requiring_api, empty_judgment_count, fully_annotated,
            partially_annotated, duplicate_case_id_count, duplicate_judgment_count, case_type_undetectable)
VALIDATION_SUMMARY_SHOWN = True


INPUT VALIDATION SUMMARY
Input filename              : criminal_batch_002.csv
Output directory            : final/Criminal
Total rows                  : 20
Columns found               : case_id, language, case_type, judgment_text, subject, object, objective_aspect, subjective_aspect, legal_provision, reasoning
Missing columns created     : none
Case type distribution (raw): {'Criminal': 20}
Case type (detected)        : {'Criminal': 20}
Language distribution       : {'Hindi': 20}
Empty judgment_text rows    : 0
Fully annotated rows        : 0
Partially annotated rows    : 19
Rows requiring API annotation: 20 (mode: fill_missing)
Case type undetectable rows : 0
Duplicate case_id rows      : 0
Exact duplicate judgments   : 0


In [132]:
# CELL 23: Checkpoint initialization and resume logic
assert VALIDATION_SUMMARY_SHOWN, "Run the validation summary cell before annotation."

def _checkpoint_meta(frame):
    ids_digest = hashlib.md5("\n".join(sorted(frame["_internal_row_id"])).encode("utf-8")).hexdigest()
    return {
        "batch_stem": BATCH_STEM,
        "input_file": str(INPUT_PATH),
        "row_count": len(frame),
        "ids_digest": ids_digest,
        "model": MODEL_NAME,
        "saved_at": now_iso(),
    }

def save_checkpoint():
    """Atomically save the full working DataFrame (with metadata) + meta sidecar."""
    if DRY_RUN:
        return
    atomic_save_csv(df, CHECKPOINT_PATH)
    atomic_write_text(CHECKPOINT_META_PATH, json.dumps(_checkpoint_meta(df), indent=2))
    logger.info("Checkpoint saved (%d rows) -> %s", len(df), CHECKPOINT_PATH.name)

RESUMED = False
if CHECKPOINT_PATH.exists():
    try:
        cp = pd.read_csv(CHECKPOINT_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        meta = {}
        if CHECKPOINT_META_PATH.exists():
            meta = json.loads(CHECKPOINT_META_PATH.read_text(encoding="utf-8"))
        current_digest = _checkpoint_meta(df)["ids_digest"]
        same_batch = (
            "_internal_row_id" in cp.columns
            and meta.get("batch_stem") == BATCH_STEM
            and meta.get("ids_digest") == current_digest
        )
        if same_batch:
            cp = cp.set_index("_internal_row_id")
            dfi = df.set_index("_internal_row_id", drop=False)
            common = dfi.index.intersection(cp.index)
            for col in TARGET_COLUMNS + RESTORE_METADATA_COLUMNS:
                if col in cp.columns:
                    dfi.loc[common, col] = cp.loc[common, col]
            df = dfi.reset_index(drop=True)
            if RETRY_FAILED_ROWS:
                retry_mask = df["_annotation_status"].isin(["failed", "failed_case_type_detection"])
                n_retry = int(retry_mask.sum())
                if n_retry:
                    df.loc[retry_mask, "_annotation_status"] = "pending"  # attempts/error history preserved
                    logger.info("Resume: %d previously failed rows reset to pending (RETRY_FAILED_ROWS=True).", n_retry)
            RESUMED = True
            _sc = df["_annotation_status"].value_counts().to_dict()
            logger.info("Resumed from checkpoint. Status counts: %s", _sc)
            print("Resumed from checkpoint:", CHECKPOINT_PATH.name)
            print("Status counts after resume:", _sc)
        else:
            backup = Path(str(CHECKPOINT_PATH) + ".bak_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
            shutil.move(str(CHECKPOINT_PATH), str(backup))
            logger.warning("Existing checkpoint does not match this input batch - backed up to %s and starting fresh.", backup.name)
            print("WARNING: checkpoint did not match this input batch. Backed up to", backup.name, "- starting fresh.")
    except Exception as exc:
        backup = Path(str(CHECKPOINT_PATH) + ".corrupt_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
        try:
            shutil.move(str(CHECKPOINT_PATH), str(backup))
        except OSError:
            pass
        logger.exception("Could not load checkpoint (%s) - backed up and starting fresh.", exc)
        print("WARNING: checkpoint unreadable (", exc, ") - backed up and starting fresh.")

if not RESUMED:
    print("No usable checkpoint - starting fresh.")


No usable checkpoint - starting fresh.


In [133]:
# CELL 24: Main annotation loop
API_CALL_COUNT = 0
RUN_START_TIME = time.time()
RUN_STATS = {"attempted": 0, "success": 0, "failed": 0, "skipped_empty_text": 0,
             "skipped_complete": 0, "reused_duplicate": 0, "failed_case_type_detection": 0,
             "retried": 0}

def _mark(idx, status, error=""):
    df.at[idx, "_annotation_status"] = status
    df.at[idx, "_annotation_error"] = safe_str(error)[:500]
    df.at[idx, "_annotation_timestamp"] = now_iso()

def _fields_needed(row):
    if ANNOTATION_MODE == "overwrite":
        return list(TARGET_COLUMNS)
    return [f for f in TARGET_COLUMNS if is_missing(row[f])]

# Duplicate-reuse map: normalized judgment hash -> the six annotation values,
# seeded from rows already successful (e.g. restored from a checkpoint).
reuse_map = {}
for _idx in df.index:
    _r = df.loc[_idx]
    if _r["_annotation_status"] in ("success", "reused_duplicate") and _r["_judgment_hash"]:
        reuse_map.setdefault(_r["_judgment_hash"], {f: _r[f] for f in TARGET_COLUMNS})

if DRY_RUN:
    print("DRY RUN - building one example prompt; no API calls, no data changes.\n")
    example_shown = False
    for idx in df.index:
        row = df.loc[idx]
        if is_missing(row["judgment_text"]):
            continue
        ctype = normalize_case_type(row["_detected_case_type"])
        if ctype is None:
            continue
        fields = _fields_needed(row)
        if not fields:
            continue
        prepared, truncated = prepare_judgment_text(row["judgment_text"])
        sp = build_system_prompt(ctype)
        up = build_user_prompt(row["case_id"], row["language"], ctype, fields, prepared)
        present_blocks = [c for c in CANONICAL_CASE_TYPES if ("CASE-TYPE DEFINITIONS: " + c.upper()) in sp]
        absent_blocks = [c for c in CANONICAL_CASE_TYPES if c not in present_blocks]
        print("Example row            :", row["_internal_row_id"], "| case_id:", row["case_id"])
        print("Detected case type     :", ctype)
        print("Fields to generate     :", ", ".join(fields))
        print("Judgment truncated     :", truncated,
              "(" + str(len(safe_str(row["judgment_text"]))) + " chars -> " + str(len(prepared)) + " chars)" if truncated else "")
        print("Definition blocks IN   :", present_blocks)
        print("Definition blocks OUT  :", absent_blocks)
        print("\n--- SYSTEM PROMPT (truncated display) " + "-" * 30)
        print(truncate_for_display(sp, 2000))
        print("\n--- USER PROMPT (truncated display) " + "-" * 32)
        print(truncate_for_display(up, 1200))
        example_shown = True
        break
    if not example_shown:
        print("DRY_RUN: no eligible rows found (all rows are empty, complete, or undetectable case type).")
    print("\nDRY RUN complete. Set DRY_RUN = False to annotate.")
    logger.info("DRY RUN completed - no API calls made, no values modified.")
else:
    api_rows_processed = 0
    pbar = tqdm(total=len(df), desc="Annotating", unit="row")
    try:
        for idx in df.index:
            pbar.update(1)
            row = df.loc[idx]
            status = row["_annotation_status"]
            if status in ("success", "reused_duplicate", "skipped_complete", "skipped_empty_text"):
                continue  # resume: never re-annotate completed rows
            if status in ("failed", "failed_case_type_detection") and not RETRY_FAILED_ROWS:
                continue
            try:
                # --- per-row error boundary: nothing inside may kill the batch ---
                judgment = row["judgment_text"]
                if is_missing(judgment):
                    _mark(idx, "skipped_empty_text", "empty_or_missing_judgment_text")
                    RUN_STATS["skipped_empty_text"] += 1
                    logger.info("row=%s case_id=%s: skipped_empty_text", row["_internal_row_id"], row["case_id"])
                    continue

                ctype = normalize_case_type(row["_detected_case_type"])
                if ctype is None:
                    _mark(idx, "failed_case_type_detection",
                          "could_not_determine_case_type (raw=" + repr(safe_str(row["case_type"])) + ")")
                    RUN_STATS["failed_case_type_detection"] += 1
                    logger.warning("row=%s case_id=%s: failed_case_type_detection", row["_internal_row_id"], row["case_id"])
                    continue

                fields_to_generate = _fields_needed(row)
                if not fields_to_generate:
                    _mark(idx, "skipped_complete", "")
                    RUN_STATS["skipped_complete"] += 1
                    continue

                jhash = row["_judgment_hash"]
                if REUSE_DUPLICATE_ANNOTATIONS and jhash and jhash in reuse_map:
                    source_values = reuse_map[jhash]
                    for f in fields_to_generate:
                        df.at[idx, f] = source_values.get(f, "")
                    df.at[idx, "_model_used"] = "reused_from_duplicate"
                    _mark(idx, "reused_duplicate", "")
                    RUN_STATS["reused_duplicate"] += 1
                    logger.info("row=%s case_id=%s: reused annotation from identical judgment (hash=%s)",
                                row["_internal_row_id"], row["case_id"], jhash[:12])
                    continue

                if MAX_ROWS is not None and api_rows_processed >= MAX_ROWS:
                    logger.info("MAX_ROWS=%s reached - stopping annotation loop.", MAX_ROWS)
                    break
                api_rows_processed += 1

                prepared, truncated = prepare_judgment_text(judgment)
                if truncated:
                    logger.info("row=%s case_id=%s: judgment excerpted %d -> %d chars (head/middle/tail strategy)",
                                row["_internal_row_id"], row["case_id"], len(safe_str(judgment)), len(prepared))

                system_prompt = build_system_prompt(ctype)
                user_prompt = build_user_prompt(row["case_id"], row["language"], ctype,
                                                fields_to_generate, prepared)

                result, err, attempts_used = call_groq_annotation(system_prompt, user_prompt, row["case_id"])

                df.at[idx, "_annotation_attempts"] = str(_to_int(row["_annotation_attempts"]) + attempts_used)
                df.at[idx, "_model_used"] = MODEL_NAME
                if attempts_used > 1:
                    RUN_STATS["retried"] += 1
                RUN_STATS["attempted"] += 1

                if result is not None:
                    for f in fields_to_generate:      # fill_missing: only missing fields are written;
                        df.at[idx, f] = result.get(f, "")   # existing valid annotations are never touched
                    _mark(idx, "success", "")
                    RUN_STATS["success"] += 1
                    if jhash:
                        reuse_map.setdefault(jhash, {f: df.at[idx, f] for f in TARGET_COLUMNS})
                    logger.info("row=%s case_id=%s type=%s: success (attempts=%d, fields=%s)",
                                row["_internal_row_id"], row["case_id"], ctype, attempts_used,
                                ",".join(fields_to_generate))
                else:
                    _mark(idx, "failed", err or "unknown_error")
                    RUN_STATS["failed"] += 1
                    logger.error("row=%s case_id=%s: PERMANENTLY FAILED after %d attempts: %s",
                                 row["_internal_row_id"], row["case_id"], attempts_used, err)
                    save_checkpoint()   # persist immediately after a permanent failure

                if RUN_STATS["attempted"] % SAVE_EVERY == 0:
                    save_checkpoint()
                if SLEEP_BETWEEN_REQUESTS > 0:
                    time.sleep(SLEEP_BETWEEN_REQUESTS)

            except KeyboardInterrupt:
                raise
            except Exception as row_exc:
                _mark(idx, "failed", "row_error: " + type(row_exc).__name__ + ": " + str(row_exc))
                RUN_STATS["failed"] += 1
                logger.exception("row=%s: unexpected row-level error (batch continues).", row["_internal_row_id"])
                save_checkpoint()

    except KeyboardInterrupt:
        logger.warning("KeyboardInterrupt received - checkpointing and stopping gracefully.")
        print("\nInterrupted by user. Progress checkpointed - re-run the notebook to resume.")
    finally:
        pbar.close()
        save_checkpoint()
        logger.info("Annotation loop finished. Stats: %s | API calls: %d", RUN_STATS, API_CALL_COUNT)

    print("Annotation loop done.", RUN_STATS, "| API calls:", API_CALL_COUNT)


Annotating:   0%|          | 0/20 [00:00<?, ?row/s]

2026-07-06 17:19:14,760 | WARNING  | case_id=CHG_CRI_000023 attempt 1/4: API error: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8659, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
2026-07-06 17:19:17,614 | WARNING  | case_id=CHG_CRI_000023 attempt 2/4: API error: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8659, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_

2026-07-06 17:22:01,210 | WARNING  | case_id=CHG_CRI_000027 attempt 4/4: API error: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8990, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
2026-07-06 17:22:01,213 | ERROR    | row=cid::CHG_CRI_000027 case_id=CHG_CRI_000027: PERMANENTLY FAILED after 4 attempts: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8990, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/bill

2026-07-06 17:27:44,525 | WARNING  | case_id=CHG_CRI_000035 attempt 4/4: API error: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8692, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
2026-07-06 17:27:44,528 | ERROR    | row=cid::CHG_CRI_000035 case_id=CHG_CRI_000035: PERMANENTLY FAILED after 4 attempts: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8692, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/bill

2026-07-06 17:29:42,903 | WARNING  | case_id=CHG_CRI_000041 attempt 2/4: API error: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8857, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
2026-07-06 17:29:47,981 | WARNING  | case_id=CHG_CRI_000041 attempt 3/4: API error: APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01kw4tfrspfy7ryf814kc7r1a6` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8857, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_

Annotation loop done. {'attempted': 20, 'success': 9, 'failed': 11, 'skipped_empty_text': 0, 'skipped_complete': 0, 'reused_duplicate': 0, 'failed_case_type_detection': 0, 'retried': 15} | API calls: 60


In [134]:
# CELL 25: Final validation (source-column integrity + schema check)
missing_final_cols = [c for c in FINAL_COLUMNS if c not in df.columns]
if missing_final_cols:
    raise RuntimeError("INTEGRITY ERROR: required final columns missing: " + ", ".join(missing_final_cols)
                       + ". Final output NOT saved. Checkpoint preserved at: " + str(CHECKPOINT_PATH))

_current_source = df[SOURCE_COLUMNS].reset_index(drop=True)
if not _current_source.equals(SOURCE_SNAPSHOT):
    changed = [c for c in SOURCE_COLUMNS if not _current_source[c].equals(SOURCE_SNAPSHOT[c])]
    logger.error("SOURCE COLUMN INTEGRITY VIOLATION in columns: %s", changed)
    raise RuntimeError(
        "SOURCE COLUMN INTEGRITY VIOLATION: the following source columns were modified during this run: "
        + ", ".join(changed)
        + ". The final output was NOT saved. Checkpoint data is preserved at: " + str(CHECKPOINT_PATH)
    )

print("Final validation passed: all required columns present; source columns "
      "(case_id, language, case_type, judgment_text) are byte-identical to the loaded input.")
logger.info("Final validation passed (source-column integrity intact).")


Final validation passed: all required columns present; source columns (case_id, language, case_type, judgment_text) are byte-identical to the loaded input.


In [135]:
# CELL 26: Save annotated output (never touches the original input CSV)
if DRY_RUN:
    print("DRY_RUN = True -> final annotated CSV NOT saved (no changes were made).")
else:
    output_columns = list(FINAL_COLUMNS) + [c for c in EXTRA_ORIGINAL_COLUMNS if c in df.columns]
    if KEEP_METADATA_IN_FINAL:
        output_columns += [m for m in METADATA_COLUMNS if m in df.columns]
    final_df = df[output_columns].copy()
    atomic_save_csv(final_df, ANNOTATED_PATH)
    logger.info("Final annotated CSV saved -> %s (%d rows, %d cols)",
                ANNOTATED_PATH, len(final_df), len(final_df.columns))
    print("Final annotated CSV saved:", ANNOTATED_PATH)
    print("Columns:", ", ".join(final_df.columns))


Final annotated CSV saved: final/Criminal/criminal_batch_002_annotated.csv
Columns: case_id, language, case_type, judgment_text, subject, object, objective_aspect, subjective_aspect, legal_provision, reasoning


In [136]:
# CELL 27: Save error and duplicate reports
if DRY_RUN:
    print("DRY_RUN = True -> error/duplicate reports NOT written.")
else:
    error_mask = df["_annotation_status"].isin(["failed", "failed_case_type_detection"])
    errors_df = df.loc[error_mask, ["_internal_row_id", "case_id", "case_type", "_detected_case_type",
                                    "_annotation_status", "_annotation_error",
                                    "_annotation_attempts", "_annotation_timestamp"]].copy()
    atomic_save_csv(errors_df, ERRORS_PATH)
    atomic_save_csv(DUPLICATES_REPORT_DF, DUPLICATES_PATH)
    logger.info("Error report saved (%d rows) -> %s | Duplicate report saved (%d rows) -> %s",
                len(errors_df), ERRORS_PATH.name, len(DUPLICATES_REPORT_DF), DUPLICATES_PATH.name)
    print("Error report     :", ERRORS_PATH, "(", len(errors_df), "rows )")
    print("Duplicate report :", DUPLICATES_PATH, "(", len(DUPLICATES_REPORT_DF), "rows )")


Error report     : final/Criminal/criminal_batch_002_errors.csv ( 11 rows )
Duplicate report : final/Criminal/criminal_batch_002_duplicates.csv ( 0 rows )


In [137]:
# CELL 28: Final completion report
elapsed = time.time() - RUN_START_TIME
status_counts = df["_annotation_status"].value_counts().to_dict()

print("=" * 68)
print("COMPLETION REPORT")
print("=" * 68)
print("Input file            :", INPUT_PATH)
print("Output directory      :", OUTPUT_DIR_PATH)
print("Final annotated file  :", ANNOTATED_PATH if (not DRY_RUN) else "(not saved - DRY_RUN)")
print("Checkpoint file       :", CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else "(none)")
print("Error file            :", ERRORS_PATH if ERRORS_PATH.exists() else "(none)")
print("Duplicate report      :", DUPLICATES_PATH if DUPLICATES_PATH.exists() else "(none)")
print("Run log               :", LOG_PATH)
print("-" * 68)
print("Total rows                    :", len(df))
print("Successful annotations        :", status_counts.get("success", 0))
print("Already complete (skipped)    :", status_counts.get("skipped_complete", 0))
print("Skipped empty judgments       :", status_counts.get("skipped_empty_text", 0))
print("Failed rows                   :", status_counts.get("failed", 0))
print("Case-type detection failures  :", status_counts.get("failed_case_type_detection", 0))
print("Rows still pending            :", status_counts.get("pending", 0))
print("Retried rows (needed >1 try)  :", RUN_STATS.get("retried", 0))
print("Duplicate judgments in input  :", duplicate_judgment_count)
print("Annotations reused (duplicates):", status_counts.get("reused_duplicate", 0))
print("Total API calls               :", API_CALL_COUNT)
print("Elapsed time                  :", format(elapsed, ".1f"), "seconds")
print("-" * 68)
print("Rows still missing each annotation field:")
for f in TARGET_COLUMNS:
    print("   ", f.ljust(18), ":", int(df[f].map(is_missing).sum()))
print("=" * 68)
logger.info("COMPLETION | statuses=%s api_calls=%d elapsed=%.1fs", status_counts, API_CALL_COUNT, elapsed)


COMPLETION REPORT
Input file            : batch_data/Criminal/criminal_batch_002.csv
Output directory      : final/Criminal
Final annotated file  : final/Criminal/criminal_batch_002_annotated.csv
Checkpoint file       : final/Criminal/criminal_batch_002_checkpoint.csv
Error file            : final/Criminal/criminal_batch_002_errors.csv
Duplicate report      : final/Criminal/criminal_batch_002_duplicates.csv
Run log               : final/Criminal/criminal_batch_002_run.log
--------------------------------------------------------------------
Total rows                    : 20
Successful annotations        : 9
Already complete (skipped)    : 0
Skipped empty judgments       : 0
Failed rows                   : 11
Case-type detection failures  : 0
Rows still pending            : 0
Retried rows (needed >1 try)  : 15
Duplicate judgments in input  : 0
Annotations reused (duplicates): 0
Total API calls               : 60
Elapsed time                  : 659.9 seconds
-----------------------------

## Self-test (no API calls)

The cell below tests the helper functions — case-type normalization, conditional prompt injection, missing-value detection, robust JSON parsing (clean / fenced / prefixed / suffixed / nulls / missing keys / extra keys / invalid), and the long-judgment handler. It never calls the API and never touches your data. Run it any time to verify the pipeline plumbing.


In [138]:
# CELL 30: Self-test of helper functions (no API, no data changes)
def run_self_tests():
    # --- is_missing ---
    assert is_missing(None) and is_missing(float("nan")) and is_missing("") and is_missing("   ")
    assert is_missing("nan") and is_missing("NaN") and is_missing("none") and is_missing("NULL")
    assert not is_missing("Ram Kumar") and not is_missing("0") and not is_missing("Not Clearly Specified")

    # --- case-type normalization ---
    for v in ("Criminal", "criminal", "CRIMINAL", " criminal ", "Criminal Law"):
        assert normalize_case_type(v) == "Criminal", v
    assert normalize_case_type("Civil") == "Civil"
    assert normalize_case_type("constitutional") == "Constitutional"
    assert normalize_case_type("ADMIN") == "Administrative"
    assert normalize_case_type("Maritime") is None
    assert normalize_case_type("") is None and normalize_case_type(None) is None

    # --- fallback detection chain ---
    assert detect_row_case_type("civil") == "Civil"
    assert detect_row_case_type("", "Criminal", None, None) == "Criminal"
    assert detect_row_case_type("", None, "Constitutional", None) == "Constitutional"
    assert detect_row_case_type("", None, "batch_data", "administrative_batch_001") == "Administrative"
    assert detect_row_case_type("", None, "batch_data", "mystery_001") is None

    # --- conditional definition injection: each prompt has ONLY its own definitions ---
    for ct in CANONICAL_CASE_TYPES:
        sp = build_system_prompt(ct)
        assert "UNIVERSAL ANNOTATION PRINCIPLES" in sp
        assert "LEGAL PROVISION FIELD" in sp
        assert "REASONING FIELD" in sp
        assert "OUTPUT FORMAT (STRICT)" in sp
        assert ("CASE-TYPE DEFINITIONS: " + ct.upper()) in sp
        for other in CANONICAL_CASE_TYPES:
            if other != ct:
                assert ("CASE-TYPE DEFINITIONS: " + other.upper()) not in sp, (ct, other)
    try:
        build_system_prompt("Maritime")
        raise AssertionError("expected ValueError for unsupported case type")
    except ValueError:
        pass

    # --- robust JSON parsing ---
    base = {"subject": "A", "object": "B", "objective_aspect": "C",
            "subjective_aspect": "D", "legal_provision": "Section 302 IPC", "reasoning": "R"}
    clean = json.dumps(base)
    assert robust_parse_model_output(clean) == base                                   # clean JSON
    fenced = "```json\n" + clean + "\n```"
    assert robust_parse_model_output(fenced) == base                                  # fenced JSON
    prefixed = "Here is the annotation you asked for:\n" + clean
    assert robust_parse_model_output(prefixed) == base                                # prefix text + JSON
    suffixed = clean + "\nI hope this helps! Let me know."
    assert robust_parse_model_output(suffixed) == base                                # JSON + suffix text
    both = "Sure!\n```json\n" + clean + "\n```\nDone."
    assert robust_parse_model_output(both) == base                                    # prefix + fence + suffix
    nulls = json.dumps({**base, "subjective_aspect": None})
    assert robust_parse_model_output(nulls)["subjective_aspect"] == ""                # null -> ""
    missing = json.dumps({"subject": "A", "reasoning": "R"})
    parsed_missing = robust_parse_model_output(missing)
    assert parsed_missing["subject"] == "A" and parsed_missing["object"] == ""        # missing keys -> ""
    extra = json.dumps({**base, "confidence": 0.9, "notes": "x"})
    assert robust_parse_model_output(extra) == base                                   # extra keys ignored
    single_quoted = str(base)                                                         # Python-style single quotes
    assert robust_parse_model_output(single_quoted) == base
    trailing = clean[:-1] + ",}"
    assert robust_parse_model_output(trailing) == base                                # trailing comma repaired
    assert robust_parse_model_output("I cannot annotate this judgment.") is None      # invalid response
    assert robust_parse_model_output("") is None and robust_parse_model_output(None) is None
    assert robust_parse_model_output(json.dumps({"foo": 1, "bar": 2})) is None        # wrong schema entirely

    # --- long judgment handler ---
    short = "A short judgment."
    out, trunc = prepare_judgment_text(short, 24000)
    assert out == short and trunc is False
    long_text = ("FACTS " * 800) + ("EVIDENCE " * 800) + ("ORDER " * 800)
    out, trunc = prepare_judgment_text(long_text, 5000)
    assert trunc is True and len(out) <= 5200
    assert out.startswith("[NOTE TO ANNOTATOR MODEL") and "OMITTED DUE TO LENGTH" in out
    assert "FACTS" in out and "ORDER" in out                                          # head and tail preserved

    # --- misc helpers ---
    assert judgment_hash("  The   COURT held ") == judgment_hash("the court held")
    assert judgment_hash("") == "" and judgment_hash(None) == ""
    assert _to_int("3") == 3 and _to_int("") == 0 and _to_int(None) == 0

    print("ALL SELF-TESTS PASSED - helper pipeline verified without any API call.")

run_self_tests()


ALL SELF-TESTS PASSED - helper pipeline verified without any API call.
